In [2]:
import pandas as pd
import ast

#When I first loaded the credits csv there were ~1600 blank columns.  Pandas had a hard time determining what were legit cols becasue of the comma placements.
#instead I wanted just three cols that can be searched. 
credits_df = pd.read_csv("Datasets/credits.csv", usecols=[0,1,2], engine="python")
credits_df.columns = ["cast", "crew", "id"]
# credits_df.to_csv("clean_credits.csv", index=False)
keyword_df = pd.read_csv("Datasets/keywords.csv")
movies_df = pd.read_csv("Datasets/movies_metadata.csv")


/var/folders/km/h432qbx55ljb5h2m9n7_s65c0000gn/T/ipykernel_21067/1739473620.py:10: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_df = pd.read_csv("Datasets/movies_metadata.csv")


In [28]:
credits_df.head()

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


In [29]:
credits_df

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862
...,...,...,...
45499,"[{'cast_id': 0, 'character': '', 'credit_id': ...","[{'credit_id': '5894a97d925141426c00818c', 'de...",439050
45500,"[{'cast_id': 1002, 'character': 'Sister Angela...","[{'credit_id': '52fe4af1c3a36847f81e9b15', 'de...",111109
45501,"[{'cast_id': 6, 'character': 'Emily Shaw', 'cr...","[{'credit_id': '52fe4776c3a368484e0c8387', 'de...",67758
45502,"[{'cast_id': 2, 'character': '', 'credit_id': ...","[{'credit_id': '533bccebc3a36844cf0011a7', 'de...",227506


In [30]:
keyword_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46419 entries, 0 to 46418
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        46419 non-null  int64 
 1   keywords  46419 non-null  object
dtypes: int64(1), object(1)
memory usage: 725.4+ KB


In [31]:
keyword_df.head()

,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."


In [3]:

#there are three rows that are causing problems.  There was the wrong data type in several rows that were gumming up the works. 
#I identified these rows by the junk data that was in the first col and then dropped these rows.
junk_data = movies_df[~movies_df['adult'].isin(["TRUE", "FALSE"])].index

movies_df.drop(index=junk_data, inplace=True)

In [4]:
movies_df['id'] = movies_df['id'].astype(int)
keyword_df['id'] = keyword_df['id'].astype(int)
movies_df = pd.merge(movies_df, keyword_df, on='id')

In [5]:
movies_df['movie_text'] = (
    # movies_df['production_companies'].fillna('') + " " + 
    movies_df['genres'].fillna('') + " " + 
    movies_df['tagline'].fillna('') + " " + 
    movies_df['keywords'].fillna('') + " " + 
    movies_df['overview'].fillna('')
)

In [6]:
import re

def clean_movie_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"'id': \d+", '', text)  
    text = re.sub(r"\bname\b", '', text, flags=re.IGNORECASE)
    text = re.sub(r"[{}\[\]'\",:]", ' ', text)
    return ' '.join(text.split()).lower()

movies_df['movie_text'] = movies_df['movie_text'].apply(clean_movie_text)

In [7]:
movies_df['movie_text']

0        animation comedy family jealousy toy boy frien...
1        adventure fantasy family roll the dice and unl...
2        romance comedy still yelling. still fighting. ...
3        comedy drama romance friends are the people wh...
4        comedy just when his world is back to normal.....
                               ...                        
46477    drama family rising and falling between a man ...
46478    drama artist play pinoy an artist struggles to...
46479    action drama thriller a deadly game of wits. w...
46480    action comedy fantasy science fiction nazis vr...
46481    action comedy promises are made in heaven - bu...
Name: movie_text, Length: 46482, dtype: object

In [8]:
#Perhaps there was a better way to clean up the data from the csv that I downloaded.
#At some point I was ready to throw my laptop out of the windwow. 
#Instead I turned to Google Gemini to assist with getting a clean copy of the csv so that I can work with it in my project.
#Even this took about 20 iterations. :(
#I went through a few different approaches then decided I wanted to break everything out into separate dataframes and connect to the movie by id

def expand_movie_data(df, column_name):
    # 1. Create a copy with only needed columns
    new_df = df[['id', column_name]].copy()
    
    # 2. Convert strings to lists of dicts safely
    def safe_parse(x):
        if isinstance(x, str):
            try:
                # This handles the SyntaxError by returning an empty list if parsing fails
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return []
        return x

    new_df[column_name] = new_df[column_name].apply(safe_parse)
    
    # 3. Explode the lists into separate rows
    new_df = new_df.explode(column_name)
    
    # 4. Extract only the 'name' from each dictionary
    new_df[column_name] = new_df[column_name].apply(
        lambda x: x.get('name') if isinstance(x, dict) else None
    )
    
    # 5. Drop any rows that couldn't be parsed
    return new_df.dropna(subset=[column_name])

# Create your new dataframes
# Use movies_df for keywords if keyword_df wasn't pre-defined
keyword_df = expand_movie_data(keyword_df, 'keywords')
genres_df = expand_movie_data(movies_df, 'genres')
production_df = expand_movie_data(movies_df, 'production_companies')
# crew_df = expand_movie_data(credits_df, 'crew')
# cast_df = expand_movie_data(credits_df, 'cast')


In [9]:
#i messed something up along the way and realized that my cast and crew didn't have character, job or department attributes anymore
def expand_credits_data(df, column_name, keys_to_extract):
    """
    df: The dataframe containing the 'id' and the JSON string column
    column_name: 'cast' or 'crew'
    keys_to_extract: Dictionary mapping JSON keys to your desired column names
    """
    
    # 1. Safely parse the stringified list of dicts
    def safe_parse(x):
        try:
            return ast.literal_eval(x) if isinstance(x, str) else x
        except (ValueError, SyntaxError):
            return []

    # 2. Prepare a list to hold the new rows
    expanded_rows = []
    
    for _, row in df.iterrows():
        movie_id = row['id']
        items = safe_parse(row[column_name])
        
        if isinstance(items, list):
            for item in items:
                # Build a dictionary for the new row starting with the movie id
                new_row = {'movie_id': movie_id}
                # Add the specific fields you requested
                for json_key, new_col_name in keys_to_extract.items():
                    new_row[new_col_name] = item.get(json_key)
                expanded_rows.append(new_row)
    
    return pd.DataFrame(expanded_rows)

# --- EXECUTION ---

# For Cast: actor_name and character
cast_df = expand_credits_data(credits_df, 'cast', {
    'name': 'actor_name', 
    'character': 'character'
})

# For Crew: crew_name, job, and department
crew_df = expand_credits_data(credits_df, 'crew', {
    'name': 'crew_name', 
    'job': 'job', 
    'department': 'department'
})


In [39]:
keyword_df

,id,keywords
0,862,jealousy
0,862,toy
0,862,boy
0,862,friendship
0,862,friends
...,...,...
46411,289923,mockumentary
46414,439050,tragic love
46415,111109,artist
46415,111109,play


In [40]:
genres_df

,id,genres
0,862,Animation
0,862,Comedy
0,862,Family
1,8844,Adventure
1,8844,Fantasy
...,...,...
46480,302349,Comedy
46480,302349,Fantasy
46480,302349,Science Fiction
46481,47871,Action


In [41]:
production_df

,id,production_companies
0,862,Pixar Animation Studios
1,8844,TriStar Pictures
1,8844,Teitler Film
1,8844,Interscope Communications
2,15602,Warner Bros.
...,...,...
46478,111109,Sine Olivia
46479,67758,American World Pictures
46480,302349,27 Films Production
46480,302349,Potemkino


In [42]:
movies_df

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,keywords,movie_text
0,FALSE,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...",pixar animation studios animation comedy famil...
1,FALSE,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,"[{'id': 10090, 'name': 'board game'}, {'id': 1...",tristar pictures teitler film interscope commu...
2,FALSE,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392...",warner bros. lancaster gate romance comedy sti...
3,FALSE,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0,"[{'id': 818, 'name': 'based on novel'}, {'id':...",twentieth century fox film corporation comedy ...
4,FALSE,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...",sandollar productions touchstone pictures come...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46477,FALSE,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",http://www.imdb.com/title/tt6209470/,439050,tt6209470,fa,رگ خواب,Rising and falling between a man and woman.,...,90.0,"[{'iso_639_1': 'fa', 'name': 'فارسی'}]",Released,Rising and falling between a man and woman,Subdue,False,4.0,1.0,"[{'id': 10703, 'name': 'tragic love'}]",drama family rising and falling between a man ...
46478,FALSE,NaN,0,"[{'id': 18, 'name': 'Drama'}]",NaN,111109,tt2028550,tl,Siglo ng Pagluluwal,An artist struggles to finish his work while a...,...,360.0,"[{'iso_639_1': 'tl', 'name': ''}]",Released,NaN,Century of Birthing,False,9.0,3.0,"[{'id': 2679, 'name': 'artist'}, {'id': 14531,...",sine olivia drama artist play pinoy an artist ...
46479,FALSE,NaN,0,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",NaN,67758,tt0303758,en,Betrayal,"When one of her hits goes wrong, a professiona...",...,90.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,A deadly game of wits.,Betrayal,False,3.8,6.0,[],american world pictures action drama thriller ...
46480,FALSE,"{'id': 312977, 'name': 'Iron Sky Collection', ...",18000000,"[{'id': 28, 'name': 'Action'}, {'id': 35, 'nam...",http://www.ironsky.net/,302349,tt3038708,en,Iron Sky: The Coming Race,"Twenty years after the events of Iron Sky, the...",...,0.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Post Production,NaN,Iron Sky: The Coming Race,False,0.0,0.0,"[{'id': 2652, 'name': 'nazis'}, {'id': 226448,...",27 films production potemkino iron sky univers...


In [10]:
cast_df.to_csv('Datasets/clean_cast.csv', index=False)

crew_df.to_csv('Datasets/clean_crew.csv', index=False)

movies_df.to_csv('Datasets/clean_movies.csv', index=False)

keyword_df.to_csv('Datasets/clean_keyword.csv', index=False)

production_df.to_csv('Datasets/clean_production.csv', index=False)

genres_df.to_csv('Datasets/clean_genres.csv', index=False)


In [46]:
cast_df

,movie_id,actor_name,character
0,862,Tom Hanks,Woody (voice)
1,862,Tim Allen,Buzz Lightyear (voice)
2,862,Don Rickles,Mr. Potato Head (voice)
3,862,Jim Varney,Slinky Dog (voice)
4,862,Wallace Shawn,Rex (voice)
...,...,...,...
560257,227506,Iwan Mosschuchin,
560258,227506,Nathalie Lissenko,
560259,227506,Pavel Pavlov,
560260,227506,Aleksandr Chabrov,


In [44]:
crew_df

,movie_id,crew_name,job,department
0,862,John Lasseter,Director,Directing
1,862,Joss Whedon,Screenplay,Writing
2,862,Andrew Stanton,Screenplay,Writing
3,862,Joel Cohen,Screenplay,Writing
4,862,Alec Sokolow,Screenplay,Writing
...,...,...,...,...
459722,67758,Richard McHugh,Original Music Composer,Sound
459723,67758,João Fernandes,Director of Photography,Camera
459724,227506,Yakov Protazanov,Director,Directing
459725,227506,Joseph N. Ermolieff,Producer,Production
